# Visium HD - SCA003

## Import packages

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json
import gc

from dask_image.imread import imread

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
from _visium_hd import visium_hd

## Create sdata

In [4]:
# TODO: Visium HD reader in harpy currently does not work due to bug fix in spatialdata-io (https://github.com/scverse/spatialdata-io/pull/303). Update harpy reader.
# TODO: Discuss option with Arne to overwrite tissue positions with our own registered coordinates.
# TODO: Rebin data based on table layer
# TODO: BMK reader, Nova-ST reader, Curio reader, StereoSeq reader

In [5]:
path_outs = r'D:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\SpaceRanger\outs'
path_img = r'D:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\2\SCA003_reseq\IMAGES\SCA003_HE_hires.tif'
dataset_id = 'SCA003'
output_path = r'D:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\harpy\sdata.zarr'

In [6]:
sdata = visium_hd(
    path=path_outs, 
    dataset_id=dataset_id,
    filtered_counts_file=False,
    bin_size=2, 
    bins_as_squares=True, 
    fullres_image_file=path_img,
    load_all_images=True,
    output=output_path
)

sdata

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
<tifffile.TiffFile 'SCA003_HE_hires.tif'> ImageJ series metadata invalid or corrupted file


INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           
INFO     The instance_key column in 'table.obs' ('table.obs[location_id]') will be relabeled to ensure a numeric   
         data type, with a continuous range and without including the value 0 (which is reserved for the           
         background). The new labels will be stored in a new column named 'relabeled_location_id'.                 


c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\spatialdata.py:511: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(table)
<tifffile.TiffFile 'SCA003_HE_hires.tif'> ImageJ series metadata invalid or corrupted file


INFO     The Zarr backing store has been changed from None the new file path:                                      
         D:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\harpy\sdata.zarr                               


version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
c:\Users\julien.mortier\conda-envs\harpy\lib

SpatialData object, with associated Zarr store: D:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\SCA003\harpy\sdata.zarr
├── Images
│     ├── 'SCA003_cytassist_image': DataArray[cyx] (3, 3000, 3203)
│     ├── 'SCA003_full_image': DataTree[cyx] (3, 15241, 16021), (3, 7620, 8010), (3, 3810, 4005), (3, 1905, 2002), (3, 952, 1001)
│     ├── 'SCA003_hires_image': DataArray[cyx] (3, 5708, 6000)
│     └── 'SCA003_lowres_image': DataArray[cyx] (3, 571, 600)
├── Labels
│     └── 'SCA003_square_002um_labels': DataArray[yx] (3350, 3350)
├── Shapes
│     └── 'SCA003_square_002um': GeoDataFrame shape: (7520279, 1) (2D shapes)
└── Tables
      └── 'square_002um': AnnData (7520279, 32285)
with coordinate systems:
    ▸ 'SCA003', with elements:
        SCA003_cytassist_image (Images), SCA003_full_image (Images), SCA003_hires_image (Images), SCA003_lowres_image (Images), SCA003_square_002um_labels (Labels), SCA003_square_002um (Shapes)
    ▸ 'SCA003_downscaled_hires', with elements:
        SCA003_h

In [ ]:
gc.collect()

In [ ]:
import xarray
from spatialdata.transformations import Identity
xa: xarray.DataArray = sdata.images['SCA003_full_image'][f'scale0']['image']
shape = (xa.shape[1], xa.shape[2])

lvl = 100
sdata = hp.im.add_grid_labels_layer(
    sdata, 
    shape=shape, 
    size=lvl/2,
    output_shapes_layer=f"shapes_hex_{lvl}um", 
    output_labels_layer=f"labels_hex_{lvl}um", 
    grid_type='hexagon', 
    offset=(0, 0), 
    chunks=1024, 
    client=None, 
    transformations={'SCA003': Identity()}, 
    scale_factors=(2, 2, 2, 2),
    overwrite=True
)

gc.collect()

sdata = hp.tb.bin_counts( 
    sdata,
    table_layer="square_002um",
    labels_layer=f"labels_hex_{lvl}um",
    output_layer="table_custom_bins",
    to_coordinate_system="SCA003",
    append=False,
    overwrite=True,
)

gc.collect()

sdata.pl.render_images(
    element='SCA003_hires_image',
).pl.render_labels(
    element=f"labels_hex_{lvl}um", fill_alpha=0.2, contour_px=5, outline_alpha=1, color='Rp1'
).pl.show(
    coordinate_systems="SCA003",
    dpi=300,
    colorbar=False,
)

In [ ]:
gc.collect()
sdata.pl.render_images(
    element='SCA003_full_image',
).pl.render_labels(
    element=f"labels_hex_{lvl}um", fill_alpha=0.2, contour_px=5, outline_alpha=1, color='Rp1'
).pl.show(
    coordinate_systems="SCA003",
    dpi=300,
    colorbar=False,
)

In [ ]:
probe_set_df = pd.read_csv(r"e:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\2\SCA003_reseq\COUNTS\outs\probe_set.csv", comment='#')
included = set(probe_set_df[probe_set_df['included'] == True]['gene_id'])
adata = sdata.tables['square_002um'][:, sdata.tables['square_002um'].var['gene_ids'].isin(included)]

In [ ]:
print(f"Number of spots in tissue: {(adata.obs['in_tissue'] == 1).sum()}")

In [ ]:
print(f"Total number of genes: {(adata.X.sum(axis=0) > 0).sum()}")

In [ ]:
print(f"Number of genes in tissue: {(adata[adata.obs['in_tissue'] == 1].X.sum(axis=0) > 0).sum()}")

In [ ]:
print(f"Total number of UMIs: {adata.X.sum()}")

In [ ]:
print(f"Number of UMIs in tissue: {adata[adata.obs['in_tissue'] == 1].X.sum()}")

In [ ]:
included

In [ ]:
import h5py

# Open the HDF5 file
with h5py.File(r"e:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\2\SCA003_reseq\COUNTS\outs\molecule_info.h5", "r") as f:
    probe_idx = f['/probes/probe_id']
    probe_id = f['/probes/probe_id']
    count = f["/count"]

    total_sum = count[:].sum()
    total_count = count.size

    print("Sum of all numbers:", total_sum)
    print("Total number of entries:", total_count)


In [ ]:
import h5py
import numpy as np

# Open the HDF5 file
with h5py.File(r"e:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\2\SCA003_reseq\COUNTS\outs\molecule_info.h5", "r") as f:
    # Load the feature_id, feature_idx, and count datasets
    feature_id = f['/features/id'][:]
    feature_idx = f['/feature_idx'][:]
    count = f['/count'][:]

    # Decode feature_id entries
    feature_id = np.array([i.decode('ascii') for i in feature_id])
    
    # Find indices to keep
    included_feature_mask = np.isin(feature_id, list(included))
    included_feature_indices = np.where(included_feature_mask)[0]
    included_counts_mask = np.isin(feature_idx, included_feature_indices)
    
    # Sum counts for filtered indices
    included_counts = count[included_counts_mask]
    total_sum = np.sum(included_counts)
    total_count = len(included_counts)

    print("Total number of UMIs for included genes:", total_sum)
    print("Number of unique UMIs for included genes:", total_count)


In [ ]:
np.where(included_feature_mask)

In [ ]:
np.isin(feature_id, list(included))

In [ ]:
1-(total_count/total_sum)